> (중요) 여러분 구글 드라이브에 최소 7GB 이상은 확보되어 있어야 합니다!

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import unicodedata  # 0번 섹션에 추가 필요

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

> (주의) 아래 코드는 처음 딱 한 번만!

In [ ]:
import zipfile
import os
import shutil
import time

# 1. 경로 설정
dataset_zip = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/초급_프로젝트_수강생_배포용.zip")
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/")

os.makedirs(extract_path, exist_ok=True)

# 2. 메인 압축파일 해제
print(f"📦 메인 데이터셋 해제 중: {os.path.basename(dataset_zip)}")
with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 메인 압축파일 삭제 (요청사항)
os.remove(dataset_zip)
print("🗑️ 메인 압축파일 삭제 완료.")

# 3. 내부 이미지 압축파일 통합 해제 로직
print("\n🚀 이미지 폴더 통합 및 내부 압축 해제 시작...")
for file in os.listdir(extract_path):
    if file.endswith(".zip"):
        file_path = os.path.join(extract_path, file)
        
        # 이름 기반 대상 폴더 결정 (train/test 통합)
        if 'train' in file.lower():
            target_folder_name = "train_images"
        elif 'test' in file.lower():
            target_folder_name = "test_images"
        else:
            target_folder_name = file.replace(".zip", "")

        target_subfolder = os.path.join(extract_path, target_folder_name)
        os.makedirs(target_subfolder, exist_ok=True)

        print(f"📂 {file} -> {target_folder_name} 통합 중...")
        
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            for member in zip_ref.infolist():
                if not member.is_dir():
                    # 내부 경로 구조를 무시하고 파일명만 추출하여 저장
                    filename = os.path.basename(member.filename)
                    if filename:
                        target_file_path = os.path.join(target_subfolder, filename)
                        with zip_ref.open(member) as source, open(target_file_path, "wb") as target:
                            shutil.copyfileobj(source, target)
        
        # [수정 포인트] 삭제 전 잠시 대기 후 강제 삭제 시도
        try:
            time.sleep(0.5) 
            if os.path.exists(file_path):
                os.remove(file_path)
                print(f"🗑️ 삭제 성공: {file}")
        except Exception as e:
            print(f"❌ {file} 삭제 실패: {e}")

print("\n✨ 모든 작업이 완료되었습니다!")
print(f"📁 최종 데이터셋 구성: {os.listdir(extract_path)}")

- 구글 드라이브 휴지통 비우기

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

# 1. 구글 드라이브 인증
auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. 휴지통 완전히 비우기 함수
def empty_trash():
    try:
        drive_service.files().emptyTrash().execute()
        print("✅ 구글 드라이브 휴지통이 완전히 비워졌습니다.")
    except Exception as e:
        print(f"❌ 휴지통 비우기 실패: {e}")

# 실행
empty_trash()

> 압축 해제한 파일들의 반영 시간이 걸릴 수 있으므로, 커널 재시작 해주기!

#### `normalize_path`는?

`normalize_path`는 파일 경로에 포함된 **한글(유니코드) 처리 방식**을 통일하여, 경로를 찾지 못하는 에러를 방지하기 위한 함수입니다.

특히 **Google Colab**이나 **Mac, Windows** 사이에서 데이터를 주고받을 때 한글 폴더명이 깨져서 발생하는 `File Not Found` 에러를 잡는 데 필수적입니다.

<br>

##### 왜 사용하나요? (NFC vs NFD)

한글을 컴퓨터가 인식하는 방식은 크게 두 가지입니다.

* **NFC (Windows 스타일):** '강'을 '강'이라는 하나의 글자로 저장합니다.
* **NFD (Mac/Unix 스타일):** '강'을 'ㄱ', 'ㅏ', 'ㅇ'으로 쪼개서 저장합니다.

사람 눈에는 똑같이 "초급 프로젝트"라고 보이지만, 컴퓨터 입장에서는 글자 조합 방식이 다르면 **완전히 다른 경로**로 인식합니다. `normalize_path` 는 이를 **NFC(표준 방식)** 로 강제 통일해주는 역할을 합니다.

---
## 🔄 YOLOv11 + SAHI 파이프라인 (리팩토링 영역)
> 아래부터는 기존 Faster R-CNN 코드를 대체하는 새로운 파이프라인입니다.

In [ ]:
############################################################
# 7. 스마트 파일명 매칭을 적용한 COCO → YOLO 변환 스크립트
############################################################

import os
import json
import yaml
import unicodedata
import shutil

# ── 1. 경로 설정 ──
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/")
TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TEST_JSON_PATH  = os.path.join(extract_path, "merged_annotations_test_final.json")
TRAIN_IMG_DIR   = os.path.join(extract_path, "train_images")
TEST_IMG_DIR    = os.path.join(extract_path, "test_images")

YOLO_DIR = os.path.join(extract_path, "yolo_dataset")

# 🚨 기존에 꼬인 폴더 및 캐시 파일 완벽 초기화
if os.path.exists(YOLO_DIR):
    shutil.rmtree(YOLO_DIR)
for cache in ["train_images.cache", "test_images.cache", "val.cache", "train.cache"]:
    cache_path = os.path.join(extract_path, cache)
    if os.path.exists(cache_path):
        os.remove(cache_path)

# 전역 변수로 매핑 딕셔너리 관리
orig2yolo = {}
yolo2orig = {}
yolo_names = []

# ── 2. 핵심 변환 및 매칭 함수 ──
def convert_and_copy_smart(json_path, source_img_dir, output_label_dir, output_img_dir, is_train=False):
    global orig2yolo, yolo2orig, yolo_names
    os.makedirs(output_label_dir, exist_ok=True)
    os.makedirs(output_img_dir, exist_ok=True)
    
    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    # Train 데이터일 경우 73개의 클래스 정보(category_id)를 추출하여 YOLO ID(0~72)와 매핑
    if is_train:
        categories = sorted(coco["categories"], key=lambda x: x["id"])
        orig2yolo = {cat["id"]: idx for idx, cat in enumerate(categories)}
        yolo2orig = {idx: cat["id"] for idx, cat in enumerate(categories)}
        yolo_names = [cat["name"] for cat in categories]

    img_meta = {img["id"]: (img["file_name"], img["width"], img["height"]) for img in coco["images"]}
    img_anns = {}
    for ann in coco["annotations"]:
        img_anns.setdefault(ann["image_id"], []).append(ann)

    stats = {"images": 0, "annotations": 0, "not_found": 0}

    for img_id, (fname, img_w, img_h) in img_meta.items():
        base_name_from_json = os.path.splitext(fname)[0]
        
        # [핵심 로직] 물리적 파일명이 원래 이름인지, 숫자로 바뀐 이름인지 탐색
        img_path_original = os.path.join(source_img_dir, fname)
        img_path_renamed = os.path.join(source_img_dir, f"{img_id}.png")

        if os.path.exists(img_path_original):
            final_base_name = base_name_from_json
            src_img_path = img_path_original
        elif os.path.exists(img_path_renamed):
            final_base_name = str(img_id) # 1.png 라면 1.txt 로 맞춤
            src_img_path = img_path_renamed
        else:
            stats["not_found"] += 1
            continue # 둘 다 없으면 패스

        # 1) 이미지 복사
        dest_img_path = os.path.join(output_img_dir, f"{final_base_name}.png")
        if not os.path.exists(dest_img_path):
            shutil.copy(src_img_path, dest_img_path)

        # 2) 라벨 생성 (이미지 이름과 100% 동일하게)
        label_path = os.path.join(output_label_dir, f"{final_base_name}.txt")
        anns = img_anns.get(img_id, [])
        
        if not anns:
            open(label_path, "w").close()
        else:
            lines = []
            for ann in anns:
                # [수정] 원본 category_id를 YOLO ID(0~72)로 변환
                cat_id = ann["category_id"]
                yolo_id = orig2yolo.get(cat_id, 0) # Fallback handling

                x_min, y_min, bw, bh = ann["bbox"]
                x_center = max(0.0, min(1.0, (x_min + bw / 2.0) / img_w))
                y_center = max(0.0, min(1.0, (y_min + bh / 2.0) / img_h))
                w_norm   = max(0.0, min(1.0, bw / img_w))
                h_norm   = max(0.0, min(1.0, bh / img_h))
                
                # [수정] 0 대신 매핑된 yolo_id를 기록
                lines.append(f"{yolo_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
                stats["annotations"] += 1

            with open(label_path, "w", encoding="utf-8") as f:
                f.write("\n".join(lines) + "\n")
                
        stats["images"] += 1

    return stats

# ── 3. 실행 ──
print("🚀 데이터셋 매칭 및 변환 시작 (약 20~30초 소요)...")
train_stats = convert_and_copy_smart(
    TRAIN_JSON_PATH, TRAIN_IMG_DIR, 
    os.path.join(YOLO_DIR, "labels", "train"), os.path.join(YOLO_DIR, "images", "train"),
    is_train=True
)
print(f"✅ [Train] 매칭된 이미지: {train_stats['images']}장 | 어노테이션: {train_stats['annotations']}개")

val_stats = convert_and_copy_smart(
    TEST_JSON_PATH, TEST_IMG_DIR, 
    os.path.join(YOLO_DIR, "labels", "val"), os.path.join(YOLO_DIR, "images", "val"),
    is_train=False
)
print(f"✅ [Val] 매칭된 이미지: {val_stats['images']}장 | 어노테이션: {val_stats['annotations']}개")

# ── 4. data.yaml 생성 ──
# [수정] nc를 지정하고, names를 추출된 클래스 이름 리스트로 매핑
data_yaml = {
    "path": YOLO_DIR,
    "train": "images/train",
    "val":   "images/val",
    "nc":    len(yolo_names),
    "names": yolo_names,
}
yaml_path = os.path.join(YOLO_DIR, "data.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

# yolo2orig.json 파일로 매핑 정보 저장
import json
with open(os.path.join(extract_path, "yolo2orig.json"), "w") as f:
    json.dump(yolo2orig, f)
print("✅ yolo2orig.json 파일 저장 완료!")

print(f"\n✨ 다중 클래스 ({len(yolo_names)}개) 지원 가능한 완벽하게 매칭된 YOLO 데이터셋 구축 완료!")



🚀 데이터셋 매칭 및 변환 시작 (약 20~30초 소요)...
✅ [Train] 매칭된 이미지: 1489장 | 어노테이션: 4526개
✅ [Val] 매칭된 이미지: 843장 | 어노테이션: 1129개
✅ yolo2orig.json 파일 저장 완료!

✨ 다중 클래스 (73개) 지원 가능한 완벽하게 매칭된 YOLO 데이터셋 구축 완료!


In [ ]:
# ################ Model layer 구조 분석을 위한 코드입니다 실행 xxxxxxxx #################
# from ultralytics import YOLO

# # 모델 로드
# model = YOLO("yolo11m.pt")

# # 1. 요약된 구조 보기 (Layer별 정보 출력)
# model.info()

# # 2. 상세 레이어 인덱스와 이름 확인 (분류기/헤드 위치 파악용)
# print(f"{'Index':<10} | {'Name':<20} | {'Parameters':<10}")
# print("-" * 50)
# for i, (name, param) in enumerate(model.named_parameters()):
#     # 텐서 단위가 아닌 레이어 단위로 보려면 model.model.modules()를 순회해야 함
#     pass

# # 가장 확실한 레이어별 상세 리스트 출력
# for i, layer in enumerate(model.model.model):
#     print(f"Layer {i}: {layer.__class__.__name__}")

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
############################################################
# 8. Ultralytics 설치 및 YOLOv11n 학습
############################################################

!pip install -q ultralytics

from ultralytics import YOLO

# ── 모델 로드 ──
model = YOLO("yolo11x.pt")

# ── 학습 실행 ──
results = model.train(
    data=yaml_path,
    epochs=13,
    imgsz=512,
    batch=8,

    # 전이학습 & 최적화
    model="yolo11x.pt",
    freeze=23,
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    warmup_epochs=3.0,
    weight_decay=0.0005,

    # 손실 함수 가중치
    box=8.5,
    dfl=1.5,

    # 증강 제어
    close_mosaic=10,

    # 저장
    project=os.path.join(extract_path, "yolo_runs"),
    name="yolov11x_pill_detection",
    exist_ok=True,
    verbose=True,
)

BEST_PT = os.path.join(results.save_dir, "weights", "best.pt")
print(f"\n🏁 학습 완료!")
print(f"📊 best.pt: {BEST_PT}")
print(f"📊 best.pt 존재 여부: {os.path.exists(BEST_PT)}")



Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=8.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=23, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11n_pill_detection, nbs=64, nms=False, opset=None, optimize=False, optim

In [ ]:
############################################################
# 9.YOLOv11 Native 추론 (캐글 제출용 포맷팅) - final_submission_yolo11.csv 생성
############################################################

import os
import json
import pandas as pd
from ultralytics import YOLO

extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/")
TEST_IMG_DIR = os.path.join(extract_path, "test_images")
BEST_PT = os.path.join(extract_path, "yolo_runs", "yolov11x_pill_detection", "weights", "best.pt")

# ── 1. 저장해둔 yolo2orig 매핑 사전 불러오기 ──
with open(os.path.join(extract_path, "yolo2orig.json"), "r") as f:
    yolo2orig_str = json.load(f)
# JSON 키는 문자열이 되므로 사용을 위해 정수형으로 다시 변환
yolo2orig = {int(k): int(v) for k, v in yolo2orig_str.items()}

# ── 2. YOLO 모델 직접 로드 ──
model = YOLO(BEST_PT)
print(f"✅ YOLO 네이티브 모델 로드: {BEST_PT}")

test_files = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")])
print(f"📸 테스트 이미지: {len(test_files)}장\n")

# ── 3. 네이티브 추론 (SAHI 없이 전체 이미지 512 통째로 추론) ──
rows = []
for idx, fname in enumerate(test_files):
    img_path = os.path.join(TEST_IMG_DIR, fname)
    image_id = str(os.path.splitext(fname)[0])
    
    # 해상도를 512로 설정하고, 0.4 이상의 확신도만 남겨 Precision을 올립니다.
    results = model.predict(source=img_path, imgsz=512, conf=0.4, verbose=False)
    
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0].cpu().numpy())
        cls_id = int(box.cls[0].cpu().numpy())
        
        # 원본 카테고리 ID 복원 (YOLO 0번 -> 원래 알약 1899번 등)
        orig_cat = yolo2orig.get(cls_id, 1)
        
        # 💡 [핵심 수정] COCO 0번 배경 함정을 피하기 위한 다현님의 +1 로직 적용!
        category_id = orig_cat + 1
        
        rows.append({
            "image_id": image_id,
            "category_id": category_id,
            "bbox_x": round(float(x1), 2),
            "bbox_y": round(float(y1), 2),
            "bbox_w": round(float(x2 - x1), 2),
            "bbox_h": round(float(y2 - y1), 2),
            "score": round(conf, 3),
        })

    if (idx + 1) % 100 == 0 or (idx + 1) == len(test_files):
        print(f"  ⏳ {idx+1}/{len(test_files)} | 누적 탐지: {len(rows)}개")

# ── 4. DataFrame 생성 및 캐글 제출 양식 ──
print(f"\n📊 전체 탐지 완료: 총 {len(rows)}개 객체")

if rows:
    df_sub = pd.DataFrame(rows)
    # 💡 image_id를 정수형으로 임시 변환하여 완벽하게 오름차순 정렬 (캐글 에러 완벽 방어)
    df_sub["image_id_int"] = df_sub["image_id"].astype(int)
    df_sub = df_sub.sort_values(["image_id_int", "score"], ascending=[True, False]).drop(columns=["image_id_int"])
    df_sub = df_sub.reset_index(drop=True)
    
    # 일괄적으로 annotation_id 1부터 순차 부여
    df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))
    
    csv_path = os.path.join(extract_path, "final_submission_yolo11.csv")
    df_sub.to_csv(csv_path, index=False)
    print(f"\n✅ 네이티브 추론 완료: final_submission_yolo11.csv")
    print(df_sub.head())
else:
    print("⚠️ 탐지된 객체가 없습니다.")

✅ SAHI 모델 로드: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_runs/yolov11n_pill_detection/weights/best.pt
📸 테스트 이미지: 843장

  ⏳ 100/843 | 누적 탐지: 421개
  ⏳ 200/843 | 누적 탐지: 839개
  ⏳ 300/843 | 누적 탐지: 1249개
  ⏳ 400/843 | 누적 탐지: 1661개
  ⏳ 500/843 | 누적 탐지: 2086개
  ⏳ 600/843 | 누적 탐지: 2500개
  ⏳ 700/843 | 누적 탐지: 2921개
  ⏳ 800/843 | 누적 탐지: 3324개
  ⏳ 843/843 | 누적 탐지: 3508개

📊 전체 탐지 완료: 총 3508개 객체

✅ 생성 완료: final_submission.csv
📊 총 예측 객체 수: 3266
   annotation_id image_id  category_id  bbox_x  bbox_y  bbox_w  bbox_h  score
0              1        1            1     167     742     187     291   0.75
1              2        1            1     153     252     206     124   0.75
2              3        1            1     551      76     395     405   0.75
3              4        1            1     591     678     260     475   0.69
4              5       10            1     641     840     192     193   0.79
5              6       10            1     101     806     247     239   0.78
6              

In [9]:
############################################################
# 10. 모델 성능 평가 (mAP 측정)
#     (기존 baseline 셀 8의 양식을 그대로 따름)
############################################################

import json
import copy
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

if len(df_sub) == 0:
    print("⚠️ 예측 결과가 없어 mAP 평가를 건너뜁니다.")
else:
    # ── 1. Ground Truth 로드 및 단일 클래스 매핑 ──
    with open(TEST_JSON_PATH, "r", encoding="utf-8") as f:
        gt_data = json.load(f)

    gt_modified = copy.deepcopy(gt_data)
    for ann in gt_modified["annotations"]:
        ann["category_id"] = 1
    gt_modified["categories"] = [{"id": 1, "name": "pill"}]

    gt_temp_json = os.path.join(extract_path, "temp_gt_single_class.json")
    with open(gt_temp_json, "w") as f:
        json.dump(gt_modified, f)

    # ── 2. 예측 결과를 COCO 평가용 리스트로 변환 ──
    eval_results = []
    for _, row in df_sub.iterrows():
        eval_results.append({
            "image_id":    int(row["image_id"]),
            "category_id": int(row["category_id"]),
            "bbox":        [row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"]],
            "score":       float(row["score"]),
        })

    dt_temp_json = os.path.join(extract_path, "temp_eval.json")
    with open(dt_temp_json, "w") as f:
        json.dump(eval_results, f)

    print(f"📊 평가 대상: GT {len(gt_modified['annotations'])}개 vs 예측 {len(eval_results)}개\n")

    # ── 3. COCO 평가 실행 ──
    coco_gt = COCO(gt_temp_json)
    coco_dt = coco_gt.loadRes(dt_temp_json)

    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

📊 평가 대상: GT 1129개 vs 예측 3266개

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.36s).
Accumulating evaluation results...
DONE (t=0.06s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.325
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.350
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.349
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.325
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.207
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.948
 Average Recall     (AR) @[ IoU=

---
## 📋 아키텍처 변경 리포트

| 변경 항목 | 기존 베이스라인 | 변경 후 파이프라인 | 수정 이유 (Why) | 기대 효과 (Expected Impact) |
| :--- | :--- | :--- | :--- | :--- |
| **데이터 포맷** | COCO JSON (`.json`) | 정규화 TXT (`.txt`) | YOLO 아키텍처의 엔드투엔드 학습 요구사항 충족 | 병목 없는 데이터 로딩 및 파이프라인 단순화 |
| **모델 가중치** | ResNet50 (Faster R-CNN) | `yolo11m.pt` + `freeze=23` | 소규모 데이터셋 과적합 방지 및 VRAM 절약 | 기존 COCO의 엣지/형태학적 탐지 능력 보존 |
| **입력 해상도** | 기본 사이즈 | `imgsz=512`  | 소형 알약 픽셀 뭉개짐(Lost in wilderness) 방지 | 타겟 객체 분해능 향상 |
| **증강/손실** | 기본 PyTorch 설정 | `close_mosaic=10`, `box=8.5` | 알약 밀집 환경 분할 오류 제어 및 현실 객체 반영 | 훈련 막바지 자연스러운 객체 인식 및 정밀한 박싱 |
| **추론 방식** | 일반 전체 이미지 추론 | yolov11 native 추론 도입 | 압축 시 발생하는 초소형 객체 탐지 누락 원천 차단 | Kaggle 제출 Score(Recall) 및 최종 등수 극대화 |

### 추가 변경사항

1. 시작 학습률 0.01 에서 마지막 학습률 0.01 설정시 최종 학습률은 0.0001로 진행

2. 기존 SAHI 추론 기법에서 yolov11 native 추론으로 도입 (연산 속도 개선)

3. epochs 13의 이유 - warm up epochs = 3이라 epochs 3회 동안 초기 학습률 0.01. 빠른 진입후 나머지 epochs 10회를 안전하게 학습률을 올리는 방식 채택